In [5]:
from typing import TypedDict, List, Optional
class SalesReportState(TypedDict):
    request: str
    raw_data: Optional[dict]
    processed_data: Optional[dict]
    chart_config: Optional[dict]
    report: Optional[str]
    errors: List[str]
    next_action: str

In [6]:
def data_collector_agent(state: SalesReportState) -> SalesReportState:
    # Placeholder: collect raw data based on request
    # Update state with raw_data and set next_action
    return state
def data_processor_agent(state: SalesReportState) -> SalesReportState:
    # Placeholder: process raw_data and update processed_data
    # Set next_action to next step
    return state
def chart_generator_agent(state: SalesReportState) -> SalesReportState:
    # Placeholder: create chart configuration from processed_data
    # Update chart_config and set next_action
    return state
def report_generator_agent(state: SalesReportState) -> SalesReportState:
    # Placeholder: generate textual report using processed_data
    # Update report and set next_action to complete
    return state
def error_handler_agent(state: SalesReportState) -> SalesReportState:
    # Placeholder: handle errors, prepare error messages in report
    # Set next_action to complete
    return state

In [7]:
def route_next_step(state: SalesReportState) -> str:
    routing = {
        "collect": "data_collector",
        "process": "data_processor",
        "visualize": "chart_generator",
        "report": "report_generator",
        "error": "error_handler",
        "complete": "END"
    }
    return routing.get(state.get("next_action", "collect"), "END")

In [21]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, END


# ------------------------
# Shared state definition
# ------------------------
class SalesReportState(TypedDict):
    request: str
    raw_data: Optional[dict]
    processed_data: Optional[dict]
    chart_config: Optional[dict]
    report: Optional[str]
    errors: List[str]
    next_action: str


# ------------------------
# Agent functions
# ------------------------
def data_collector_agent(state: SalesReportState) -> SalesReportState:
    print("→ [Data Collector] Collecting raw data...")
    try:
        state["raw_data"] = {"sales": [100, 200, 300, 250, 400]}
        state["next_action"] = "process"  # ✅ always goes to processor
    except Exception as e:
        state["errors"].append(str(e))
        state["next_action"] = "error"
    return state


def data_processor_agent(state: SalesReportState) -> SalesReportState:
    print("→ [Data Processor] Processing data...")
    try:
        data = state["raw_data"]["sales"]
        state["processed_data"] = {"avg_sales": sum(data) / len(data)}
        state["next_action"] = "visualize"  # ✅ goes to chart next
    except Exception as e:
        state["errors"].append(str(e))
        state["next_action"] = "error"
    return state


def chart_generator_agent(state: SalesReportState) -> SalesReportState:
    print("→ [Chart Generator] Generating chart config...")
    try:
        state["chart_config"] = {
            "type": "bar",
            "labels": ["Jan", "Feb", "Mar", "Apr", "May"],
            "values": state["raw_data"]["sales"]
        }
        state["next_action"] = "report"  # ✅ goes to report next
    except Exception as e:
        state["errors"].append(str(e))
        state["next_action"] = "error"
    return state


def report_generator_agent(state: SalesReportState) -> SalesReportState:
    print("→ [Report Generator] Generating report...")
    try:
        avg = state["processed_data"]["avg_sales"]
        state["report"] = f"✅ Average sales for {state['request']}: ${avg:.2f}"
        state["next_action"] = "complete"  # ✅ end
    except Exception as e:
        state["errors"].append(str(e))
        state["next_action"] = "error"
    return state


def error_handler_agent(state: SalesReportState) -> SalesReportState:
    print("⚠️ [Error Handler] Handling errors...")
    state["report"] = "❌ Errors:\n" + "\n".join(state["errors"])
    state["next_action"] = "complete"
    return state


# ------------------------
# Router — context-aware
# ------------------------
def route_next_step(state: SalesReportState, current_node: str) -> str:
    """
    Context-aware router — knows which node is routing.
    """
    next_action = state.get("next_action", "collect")

    if current_node == "data_collector":
        if next_action == "process":
            return "data_processor"
        elif next_action == "error":
            return "error_handler"
        else:
            return END

    elif current_node == "data_processor":
        if next_action == "visualize":
            return "chart_generator"
        elif next_action == "error":
            return "error_handler"
        else:
            return END

    elif current_node == "chart_generator":
        if next_action == "report":
            return "report_generator"
        elif next_action == "error":
            return "error_handler"
        else:
            return END

    elif current_node == "report_generator":
        if next_action in ("complete", "report"):
            return END
        elif next_action == "error":
            return "error_handler"
        else:
            return END

    elif current_node == "error_handler":
        return END

    return END


# ------------------------
# Workflow graph
# ------------------------
def create_sales_report_workflow():
    workflow = StateGraph(SalesReportState)

    # Add nodes
    workflow.add_node("data_collector", data_collector_agent)
    workflow.add_node("data_processor", data_processor_agent)
    workflow.add_node("chart_generator", chart_generator_agent)
    workflow.add_node("report_generator", report_generator_agent)
    workflow.add_node("error_handler", error_handler_agent)

    # Add edges using lambda to pass current node
    workflow.add_conditional_edges("data_collector", lambda s: route_next_step(s, "data_collector"), {
        "data_processor": "data_processor",
        "error_handler": "error_handler",
        END: END
    })
    workflow.add_conditional_edges("data_processor", lambda s: route_next_step(s, "data_processor"), {
        "chart_generator": "chart_generator",
        "error_handler": "error_handler",
        END: END
    })
    workflow.add_conditional_edges("chart_generator", lambda s: route_next_step(s, "chart_generator"), {
        "report_generator": "report_generator",
        "error_handler": "error_handler",
        END: END
    })
    workflow.add_conditional_edges("report_generator", lambda s: route_next_step(s, "report_generator"), {
        "error_handler": "error_handler",
        END: END
    })
    workflow.add_conditional_edges("error_handler", lambda s: route_next_step(s, "error_handler"), {END: END})

    workflow.set_entry_point("data_collector")
    return workflow.compile()


# ------------------------
# Runner
# ------------------------
def run_sales_report_workflow():
    app = create_sales_report_workflow()
    initial_state = SalesReportState(
        request="Q1–Q2 2024 Sales Analysis",
        raw_data=None,
        processed_data=None,
        chart_config=None,
        report=None,
        errors=[],
        next_action="collect"
    )

    print("🚀 Starting workflow...\n")
    final_state = app.invoke(initial_state)
    print("\n✅ Workflow Complete\n")

    if final_state["errors"]:
        print("❌ Errors:")
        for e in final_state["errors"]:
            print(f"- {e}")

    print("\n📊 Final Report:\n", final_state["report"])
    return final_state


# ------------------------
# Main
# ------------------------
if __name__ == "__main__":
    run_sales_report_workflow()


🚀 Starting workflow...

→ [Data Collector] Collecting raw data...
→ [Data Processor] Processing data...
→ [Chart Generator] Generating chart config...
→ [Report Generator] Generating report...

✅ Workflow Complete


📊 Final Report:
 ✅ Average sales for Q1–Q2 2024 Sales Analysis: $250.00


In [2]:
pip show langgraph


Name: langgraph
Version: 1.0.2
Summary: Building stateful, multi-actor applications with LLMs
Home-page: 
Author: 
Author-email: 
License-Expression: MIT
Location: /opt/miniconda3/lib/python3.13/site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
import langgraph
dir(langgraph)


['__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__']

In [4]:
from langgraph.graph import StateGraph, END
